Imports and Setup

In [ ]:
import os
import json
import time
import pandas as pd
import google.generativeai as genai
import re
from google.generativeai.types import HarmCategory, HarmBlockThreshold

API_KEY = "API_KEY"  # Replace with your actual API key
genai.configure(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"
# Define your directories
BASE_DIR = os.getcwd()
PDF_DIRECTORY = os.path.join(BASE_DIR, "test") #put only one articles in this folder to try the code since their is a limit on the number of requests
OUTPUT_EXCEL = "scanned_pdf_extraction.xlsx"

Define the Prompt

In [10]:
EXTRACTION_PROMPT = """
**Role:** You are an expert Material Scientist and Data Curator specializing in Organic Photovoltaics (OPV).

**Task:** Extract structured device data from the provided scientific text. Identify every unique solar cell configuration and extract its fabrication details and performance metrics into a JSON format.

**Chain of Thought & Global Inheritance (CRITICAL):**
1. First, actively search the text for the "Experimental Section", "Device Fabrication", or "Materials and Methods".
2. Extract the GLOBAL processing parameters: Solvent, Additive name & amount, and Thermal Annealing (Temperature and Time).
3. You MUST apply these global parameters to ALL devices extracted, unless a specific device explicitly deviates from them in the text.

**Normalization Rules:**
* **Ratio Extraction:** Extract the D:A weight ratio clearly (e.g., "1:1.2", "1:1.2:0.2"). If an element is added by weight percentage, format it as "BaseRatio + X wt%" (e.g., "1:1.2 + 10 wt%"). If Layer-by-Layer, write "LbL". Do not write full sentences here.
* **Additives (CRITICAL DISTINCTION):** Distinguish between "missing information" and "no additive used". If the fabrication section describes the solvent but makes absolutely NO mention of any additive, or explicitly says "without additive", "w/o", "pristine", or "as-cast", you MUST output "None" for both `additive_name` and `additive_volume_percent`. ONLY use "Not reported" if the text is completely missing.
* **Solvents:** "CF" or "CHCl3" → "Chloroform" | "CB" → "Chlorobenzene" | "Tol" → "Toluene".
* **Units:** Convert efficiencies to % (e.g., 0.15 → 15%). Convert thickness to nm. Output additive amounts exactly as found (e.g., "0.35% v/v" or "50 wt%").
* **No Double Quotes in Strings (CRITICAL):** NEVER use double quotes (") inside your text values (such as in notes or global_fabrication_context). If you need to quote a word or phrase, you MUST use single quotes ('). Unescaped double quotes will break the JSON parser.

### TARGET JSON STRUCTURE
Return a single JSON object containing your evidence quote and the list of devices. Strictly adhere to this schema:

```json
{
  "global_fabrication_context": "Briefly summarize in your own words the solvent, additives, and annealing conditions. DO NOT quote exactly from the text.",
  "devices": [
    {
      "device_id": "Device 1 (e.g., Optimal Ratio 1:1.2)",
      "architecture": {
        "type": "string (Conventional, Inverted, or Unknown)",
        "anode": "string",
        "hole_transport_layer": "string",
        "electron_transport_layer": "string",
        "cathode": "string"
      },
      "active_layer": {
        "donor": "string",
        "acceptor": "string",
        "ratio_weight": "string (e.g., '1:1.2', '1:1.2 + 10 wt%', 'LbL', or 'Not reported')",
        "thickness_nm": "float or 'Not reported'"
      },
      "processing": {
        "solvent": "string (Normalized, e.g., Chloroform)",
        "total_concentration_mg_ml": "float or 'None'",
        "additive_name": "string (Normalized, or 'None')",
        "additive_volume_percent": "string ('None' or 'Not reported')",
        "deposition_method": "string",
        "annealing_temperature_celsius": "float or 'As-cast' (CRITICAL: This MUST be the post-deposition thermal annealing temperature of the ACTIVE LAYER FILM only. Do NOT confuse with solution heating temperature or transport layer annealing like ZnO/PEDOT. Put solution heating details in the notes.)",
        "annealing_time_min": "float or 'As-cast'"
      },
      "metrics": {
        "pce_percent": "float (Champion value)",
        "voc_volts": "float",
        "jsc_ma_cm2": "float",
        "ff_percent": "float",
        "HUMO": "float or 'None'",
        "LUMO": "float or 'None'"
      },
      "notes": "string (Capture nuances like 'certified PCE', 'Large Area', etc.)"
    }
  ]
}
"""

Processing Functions

In [11]:
def group_files_by_number(folder_path):
    """
    Regroupe automatiquement les fichiers PDF par numéro.
    Exemple : Article 6 + information support_Article 6
    """
    groups = {}

    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(".pdf"):
            continue

        # Cherche un numéro dans le nom
        match = re.search(r'(\d+)', filename)
        if not match:
            continue

        num = match.group(1)

        # Ajoute au groupe correspondant
        groups.setdefault(num, []).append(os.path.join(folder_path, filename))

    return groups


def process_pdf_natively(file_paths):
    model = genai.GenerativeModel(MODEL_NAME)
    uploaded_files = []

    # 1. Upload des fichiers (Inchangé)
    for file_path in file_paths:
        file_name = os.path.basename(file_path)
        print(f"  - Uploading {file_name} to Gemini...")
        try:
            uploaded = genai.upload_file(path=file_path, display_name=file_name)
            while uploaded.state.name == "PROCESSING":
                print(f"    Processing {file_name}...", end="\r")
                time.sleep(2)
                uploaded = genai.get_file(uploaded.name)
            if uploaded.state.name == "FAILED":
                continue
            uploaded_files.append(uploaded)
        except Exception as e:
            print(f"    ! Error uploading {file_name}: {e}")

    if not uploaded_files:
        return []

    print("    All files ready. Sending prompt to model...")

    # --- LA BOUCLE DE RETRY EST ICI ---
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                uploaded_files + [EXTRACTION_PROMPT],
                # --- ON DÉSACTIVE LES FILTRES DE SÉCURITÉ ---
                safety_settings={
                    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                }
            )
            
            raw_text = response.text
            
            # NOUVEAU : Affiche la raison EXACTE pour laquelle le modèle s'est arrêté
            print(f"    -> Fin de génération: {response.candidates[0].finish_reason.name}")
            
            # Nettoyeur Regex
            match = re.search(r'(\{.*\})', raw_text, re.DOTALL)
            if not match:
                match = re.search(r'(\[.*\])', raw_text, re.DOTALL)
                
            clean_text = match.group(1) if match else raw_text
            
            # Lecture du JSON
            json_data = json.loads(clean_text, strict=False)
            
            # Si on arrive ici, c'est que le JSON est VALIDE ! On sort de la boucle.
            if isinstance(json_data, dict) and "devices" in json_data:
                devices = json_data["devices"]
            elif isinstance(json_data, list):
                devices = json_data
            else:
                devices = [json_data]
                
            # Nettoyage des fichiers sur le serveur
            for f in uploaded_files:
                genai.delete_file(f.name)
                
            return devices # Extraction réussie, on renvoie les données

        except json.JSONDecodeError as e:
            # Si le JSON est cassé, on l'attrape ici
            print(f"    ! Erreur de syntaxe JSON du modèle (Tentative {attempt + 1}/{max_retries}).")
            
            if attempt == max_retries - 1: # Si c'était la dernière tentative
                print("    ! Échec définitif. Le modèle n'arrive pas à formater cet article.")
                nom_erreur = f"erreur_{os.path.basename(file_paths[0])}.txt"
                with open(nom_erreur, "w", encoding="utf-8") as f:
                    f.write(raw_text if 'raw_text' in locals() else "Pas de texte")
                
                # Nettoyage
                for f in uploaded_files:
                    try: genai.delete_file(f.name)
                    except: pass
                return []
            else:
                print("    -> Relance automatique de l'API en cours...")
                time.sleep(3) # Petite pause avant de redemander à l'IA

        except Exception as e:
            print(f"    ! Erreur inattendue : {e}")
            for f in uploaded_files:
                try: genai.delete_file(f.name)
                except: pass
            return []

def flatten_json(devices_list, filename):
    flat_rows = []
    for entry in devices_list:
        if not entry: continue
        
        proc = entry.get("processing", {})
        active = entry.get("active_layer", {})
        metrics = entry.get("metrics", {})
        arch = entry.get("architecture", {})

        row = {
            "Source File": filename,
            "Device ID": entry.get("device_id"),
            "Structure": arch.get("type"),
            "Donor": active.get("donor"),
            "Acceptor": active.get("acceptor"),
            "Ratio": active.get("ratio_weight"), # <-- Le ratio sera maintenant correct !
            "Solvent": proc.get("solvent"),
            "Additive": proc.get("additive_name"),
            "Annealing Temp": proc.get("annealing_temperature_celsius"),
            "PCE (%)": metrics.get("pce_percent"),
            "Voc (V)": metrics.get("voc_volts"),
            "Jsc (mA/cm2)": metrics.get("jsc_ma_cm2"),
            "FF (%)": metrics.get("ff_percent"),
            "Notes": entry.get("notes")
        }
        flat_rows.append(row)
    return flat_rows

Main Execution

In [12]:
# 1. Regrouper les fichiers par numéro
groups = group_files_by_number(PDF_DIRECTORY)
all_data = []

print(f"Found {len(groups)} groups in '{PDF_DIRECTORY}'")

# 2. Traiter chaque groupe
for num, file_list in groups.items():
    print(f"\nProcessing group {num} ({len(file_list)} files)")

    # 3. Envoyer les 2 fichiers à Gemini
    extracted_json = process_pdf_natively(file_list)

    if extracted_json:
        flat_data = flatten_json(extracted_json, f"Article {num}")
        all_data.extend(flat_data)
        print(f"  -> Successfully extracted {len(flat_data)} devices.")
    else:
        print("  -> No data found or error occurred.")

# 4. Export Excel
if all_data:
    df = pd.DataFrame(all_data)
    df.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\n✅ Success! Data saved to {OUTPUT_EXCEL}")
    display(df.head())
else:
    print("\n No data extracted from any groups.")


Found 10 groups in 'c:\Users\mguir\Documents\Projet\test'

Processing group 100 (2 files)
  - Uploading Article 100.pdf to Gemini...
  - Uploading information support_Article 100.pdf to Gemini...
    All files ready. Sending prompt to model...
    -> Fin de génération: STOP
  -> Successfully extracted 4 devices.

Processing group 26 (2 files)
  - Uploading Article 26.pdf to Gemini...
  - Uploading information support_Article 26.pdf to Gemini...
    All files ready. Sending prompt to model...
    -> Fin de génération: STOP
  -> Successfully extracted 34 devices.

Processing group 38 (2 files)
  - Uploading Article 38.pdf to Gemini...
  - Uploading information support_Article 38.pdf to Gemini...
    All files ready. Sending prompt to model...
    -> Fin de génération: STOP
  -> Successfully extracted 5 devices.

Processing group 48 (2 files)
  - Uploading Article 48.pdf to Gemini...
  - Uploading information support_Article 48.pdf to Gemini...
    All files ready. Sending prompt to model

,Source File,Device ID,Structure,Donor,Acceptor,Ratio,Solvent,Additive,Annealing Temp,PCE (%),Voc (V),Jsc (mA/cm2),FF (%),Notes
0,Article 100,J81:ITIC (as-cast),Conventional,J81,ITIC,1:1,Chloroform,None,As-cast,8.03,0.94,13.96,61.16,"Device based on J81:ITIC, as-cast film."
1,Article 100,J81:ITIC (annealed),Conventional,J81,ITIC,1:1,Chloroform,None,160.0,10.60,0.95,15.27,73.08,"Device based on J81:ITIC, thermally annealed a..."
2,Article 100,J81:m-ITIC (as-cast),Conventional,J81,m-ITIC,1:1,Chloroform,None,As-cast,7.21,0.95,14.62,51.90,"Device based on J81:m-ITIC, as-cast film."
3,Article 100,J81:m-ITIC (annealed),Conventional,J81,m-ITIC,1:1,Chloroform,None,160.0,11.05,0.96,16.48,69.83,"Device based on J81:m-ITIC, thermally annealed..."
4,Article 26,PBDB-T:BTP-eC9-G51 (Inverted) - Ratio 1:0.8,Inverted,PBDB-T,BTP-eC9-G51,1:0.8,Chloroform,"1,8-diiodooctane",As-cast,14.56,0.91,22.49,72.04,"PCE, Jsc values calculated by integrating EQE ..."
